# Teil 2 - Datenbeschreibung und Vorbereitung

In diesem Notebook lade ich den Smartphone-Datensatz, prüfe seine Struktur, beschreibe fehlende Werte und berechne einfache Statistiken. Danach wähle ich ein sinnvolles Vorhersagefeld, erstelle Grafiken und skaliere ein numerisches Feld mit `MinMaxScaler`. Alle Pfade sind relativ zum Repository.

## 1. Datensatz laden

Die Datei ist eine CSV-Datei mit Semikolon als Trennzeichen. Ich lade sie mit `pandas`, damit Spalten, Datentypen, fehlende Werte und Statistiken übersichtlich ausgewertet werden können.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

DATA_FILE = Path('Smartphone_Usage_Productivity_Dataset_50000.csv')
df = pd.read_csv(DATA_FILE, sep=';')

print('Zeilen:', df.shape[0])
print('Spalten:', df.shape[1])
df.head()

Zeilen: 50000
Spalten: 13


,User_ID,Age,Gender,Occupation,Device_Type,Daily_Phone_Hours,Social_Media_Hours,Work_Productivity_Score,Sleep_Hours,Stress_Level,App_Usage_Count,Caffeine_Intake_Cups,Weekend_Screen_Time_Hours
0,U1,58,Male,Professional,Android,1.3,6.7,6,8.8,4,42,1,8.7
1,U2,25,Male,Professional,Android,1.2,1.5,5,6.4,1,51,3,5.1
2,U3,19,Male,Student,iOS,5.3,5.7,5,9.0,4,14,5,6.3
3,U4,35,Female,Business Owner,iOS,5.8,2.5,2,5.7,3,36,6,12.8
4,U5,33,Male,Freelancer,Android,7.9,1.3,4,5.7,3,37,5,9.9


## 2. Datenüberblick

Hier prüfe ich Spaltennamen, Datentypen und fehlende Werte. Das ist wichtig, weil Machine-Learning-Modelle numerische Daten, saubere Zielwerte und einen bewussten Umgang mit Kategorien brauchen.

In [2]:
overview = pd.DataFrame({
    'Datentyp': df.dtypes.astype(str),
    'Fehlende Werte': df.isna().sum(),
    'Einzigartige Werte': df.nunique()
})
overview

,Datentyp,Fehlende Werte,Einzigartige Werte
User_ID,str,0,50000
Age,int64,0,43
Gender,str,0,3
Occupation,str,0,4
Device_Type,str,0,2
Daily_Phone_Hours,float64,0,111
Social_Media_Hours,float64,0,76
Work_Productivity_Score,int64,0,10
Sleep_Hours,float64,0,51
Stress_Level,int64,0,10


In [3]:
print('Beispielzeilen:')
df.sample(5, random_state=42)

Beispielzeilen:


,User_ID,Age,Gender,Occupation,Device_Type,Daily_Phone_Hours,Social_Media_Hours,Work_Productivity_Score,Sleep_Hours,Stress_Level,App_Usage_Count,Caffeine_Intake_Cups,Weekend_Screen_Time_Hours
33553,U33554,42,Male,Business Owner,Android,7.4,7.4,2,8.5,8,18,1,3.8
9427,U9428,21,Other,Professional,iOS,1.4,2.4,3,4.3,2,6,6,7.6
199,U200,43,Male,Professional,iOS,8.4,1.0,8,8.6,4,26,4,7.9
12447,U12448,45,Other,Freelancer,iOS,3.4,2.9,4,6.7,9,24,3,7.5
39489,U39490,43,Female,Business Owner,iOS,4.7,6.6,3,8.6,2,35,4,4.4


## 3. Mögliche Zielvariablen

1. `Stress_Level`: Gut geeignet, weil Stress fachlich mit Schlaf, Bildschirmzeit, Social Media und Koffein zusammenhängen kann. Für die Wahrheitsmatrix kann daraus eine klare Klasse `High_Stress` gebildet werden. Nachteil: Die ersten Korrelationen sind sehr schwach.
2. `Work_Productivity_Score`: Fachlich interessant, weil Produktivität durch Schlaf, Stress und Smartphone-Nutzung beeinflusst sein könnte. Nachteil: Die Werte 1 bis 10 wirken im Datensatz fast zufällig verteilt.
3. `Sleep_Hours`: Als Regression möglich, aber eine Wahrheitsmatrix passt dazu nicht direkt. Zudem zeigen die numerischen Felder kaum lineare Zusammenhänge.

Ich wähle `Stress_Level` als Zielfeld und bilde daraus `High_Stress = 1`, wenn `Stress_Level >= 7` ist. Diese Entscheidung ist prüfungskonform, weil es ein Klassifikationsproblem mit Confusion Matrix, Sensitivität und Spezifität ergibt. Gleichzeitig wird die Einschränkung dokumentiert: Der Datensatz enthält viele Zeilen, aber nur schwache erkennbare Muster.

In [4]:
df['High_Stress'] = (df['Stress_Level'] >= 7).astype(int)
class_distribution = df['High_Stress'].value_counts().rename(index={0: 'kein hoher Stress', 1: 'hoher Stress'})
class_distribution

High_Stress
kein hoher Stress    30050
hoher Stress         19950
Name: count, dtype: int64

## 4. Statistische Informationen

Für numerische Spalten berechne ich Mittelwert, Median, Standardabweichung, Minimum und Maximum. Für kategoriale Spalten prüfe ich Anzahl verschiedener Werte und den häufigsten Wert.

In [5]:
numeric_columns = df.select_dtypes(include=[np.number]).columns
numeric_stats = df[numeric_columns].describe().T[['mean', '50%', 'std', 'min', 'max']]
numeric_stats = numeric_stats.rename(columns={'50%': 'median'})
numeric_stats.round(2)

,mean,median,std,min,max
Age,39.03,39.0,12.41,18.0,60.0
Daily_Phone_Hours,6.51,6.5,3.17,1.0,12.0
Social_Media_Hours,4.27,4.3,2.16,0.5,8.0
Work_Productivity_Score,5.50,5.5,2.87,1.0,10.0
Sleep_Hours,6.50,6.5,1.45,4.0,9.0
Stress_Level,5.50,6.0,2.87,1.0,10.0
App_Usage_Count,32.44,32.0,16.12,5.0,60.0
Caffeine_Intake_Cups,3.00,3.0,2.00,0.0,6.0
Weekend_Screen_Time_Hours,8.01,8.0,3.46,2.0,14.0
High_Stress,0.40,0.0,0.49,0.0,1.0


In [6]:
categorical_columns = [column for column in df.columns if column not in numeric_columns]
category_stats = []
for column in categorical_columns:
    top_value = df[column].mode(dropna=True).iloc[0]
    top_count = int((df[column] == top_value).sum())
    category_stats.append({
        'Spalte': column,
        'Einzigartige Werte': df[column].nunique(),
        'Häufigster Wert': top_value,
        'Anzahl': top_count,
    })

pd.DataFrame(category_stats)

,Spalte,Einzigartige Werte,Häufigster Wert,Anzahl
0,User_ID,50000,U1,1
1,Gender,3,Male,16708
2,Occupation,4,Professional,12629
3,Device_Type,2,Android,25080


## 5. Grafiken

Das Histogramm zeigt die Verteilung der täglichen Smartphone-Nutzung. Die Korrelationsmatrix zeigt, ob numerische Felder starke lineare Zusammenhänge besitzen. Genau diese Prüfung ist hier wichtig, weil ein Modell nur gut lernen kann, wenn brauchbare Muster im Datensatz vorhanden sind.

In [7]:
plt.figure(figsize=(8, 4))
plt.hist(df['Daily_Phone_Hours'], bins=20, edgecolor='black')
plt.title('Verteilung der täglichen Smartphone-Nutzung')
plt.xlabel('Daily_Phone_Hours')
plt.ylabel('Anzahl')
plt.tight_layout()
plt.show()

/var/folders/hp/m0vq6vz143s1lhd11rqb_ft00000gn/T/ipykernel_25753/2418851493.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
corr = df[numeric_columns].corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Korrelation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Korrelationsmatrix der numerischen Felder')
plt.tight_layout()
plt.show()

corr[['Stress_Level', 'High_Stress']].sort_values('High_Stress', ascending=False).round(3)

/var/folders/hp/m0vq6vz143s1lhd11rqb_ft00000gn/T/ipykernel_25753/2871300591.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Stress_Level,High_Stress
High_Stress,0.852,1.000
Stress_Level,1.000,0.852
Weekend_Screen_Time_Hours,0.003,0.007
Sleep_Hours,0.005,0.007
Work_Productivity_Score,0.010,0.005
Social_Media_Hours,-0.003,0.002
Caffeine_Intake_Cups,0.002,0.001
Age,-0.002,-0.000
App_Usage_Count,-0.001,-0.002
Daily_Phone_Hours,-0.003,-0.004


## 6. Skalierung

Ich skaliere `Daily_Phone_Hours` mit `MinMaxScaler` auf den Bereich 0 bis 1. Das ist sinnvoll, weil die Spalte eine andere Einheit und Skala als andere numerische Felder hat. Auch wenn der später gewählte Random-Forest-Algorithmus Skalierung nicht zwingend braucht, zeigt dieser Schritt korrekt, wie numerische Daten für Modelle vorbereitet werden können.

In [9]:
scaler = MinMaxScaler()
df['Daily_Phone_Hours_scaled'] = scaler.fit_transform(df[['Daily_Phone_Hours']])

df[['Daily_Phone_Hours', 'Daily_Phone_Hours_scaled']].head(10)

,Daily_Phone_Hours,Daily_Phone_Hours_scaled
0,1.3,0.027273
1,1.2,0.018182
2,5.3,0.390909
3,5.8,0.436364
4,7.9,0.627273
5,10.9,0.900000
6,5.6,0.418182
7,8.5,0.681818
8,9.4,0.763636
9,2.8,0.163636


In [10]:
df[['Daily_Phone_Hours', 'Daily_Phone_Hours_scaled']].agg(['min', 'max', 'mean']).round(3)

,Daily_Phone_Hours,Daily_Phone_Hours_scaled
min,1.000,0.000
max,12.000,1.000
mean,6.509,0.501


## Fazit Teil 2

Der Datensatz ist gross genug, hat keine fehlenden Werte und enthält numerische sowie kategoriale Felder. Als Ziel wird `Stress_Level` verwendet und für die Klassifikation in `High_Stress` umgewandelt. Die Grafiken und Korrelationen zeigen aber, dass die Zusammenhänge im Datensatz schwach sind. Deshalb ist bei Modell und Evaluation keine sehr hohe Genauigkeit zu erwarten.